In [1]:
import dask.dataframe as dd
import locale
import os
import numpy as np
import pandas as pd
import shutil

In [2]:
# défintion de paramètres locaux fr pour obtenir les mois et jour en français
locale.setlocale(locale.LC_ALL, 'fr_FR.utf8')

'fr_FR.utf8'

In [3]:
# Création d'un dictionnaire des dtypes
dic_dtypes = {'Viaje id' : "int32",
          'Usuario_Id' : "int32",
          'Genero' : "object",
          'Año_de_nacimiento' : "float64",
          'Inicio_del_viaje' : "object",
          'Fin_del_viaje' : "object",
          'Origen_Id' : "int32",
          'Destino_Id' : "int32",
          "id" : "int32",
          "name" : "object", 
          "obcn" : "object", 
          "location" : "category",
          "latitude" : "float32",
          "longitude" : "float32",
          "status" : "object"}

In [4]:
# Charger le fichier nomenclature qui servira à la jointure et obtenir le nom des stations et leur localisation à partir de leur id
df_stations = pd.read_csv('data/nomenclatura_2025_11.csv', dtype=dic_dtypes)

# Renommer la colonne 'id' en 'station_id' pour la clarté et la jointure
df_stations = df_stations.rename(columns={'id': 'station_id', 'name':'nom', 'location':'lieu'})

# Sélectionner uniquement les colonnes nécessaires (pour éviter le bruit)
cols_a_garder = ['station_id', 'nom', 'lieu', 'latitude', 'longitude']
df_stations = df_stations[cols_a_garder]

In [5]:
df_stations.head()

,station_id,nom,lieu,latitude,longitude
0,2,(GDL-001) C. Epigmenio Glez./ Av. 16 de Sept.,POLÍGONO CENTRAL,20.666378,-103.348824
1,3,(GDL-002) C. Colonias / Av. Niños héroes,POLÍGONO CENTRAL,20.667229,-103.365997
2,4,(GDL-003) C. Vidrio / Av. Chapultepec,POLÍGONO CENTRAL,20.667690,-103.368256
3,5,(GDL-004) C. Ghilardi /C. Miraflores,POLÍGONO CENTRAL,20.691847,-103.362549
4,6,(GDL-005) C. San Diego /Calzada Independencia,POLÍGONO CENTRAL,20.681158,-103.339363


In [ ]:
all_files = 'data/202*/*.csv'

ddf = dd.read_csv(
    all_files,
    sep=',',
    encoding='ISO-8859-1',
    dtype=dic_dtypes,
    assume_missing=True # Pour être sûr que les colonnes int passent en float si besoin
)

# Renommer les colonnes
ddf.columns = ["id_trajet", "id_utilisateur", "genre", "annee_naissance", "debut_trajet", "fin_trajet", "id_depart", "id_arrivee"]

# Afficher les métadonnées pour confirmation
print("--- Dask DataFrame chargé ---")
print(ddf.head())
print(ddf.dtypes)

--- Dask DataFrame chargé ---
    id_trajet  id_utilisateur genre  annee_naissance         debut_trajet  \
0  14420217.0          451617     M           1992.0  2020-01-01 06:02:20   
1  14420218.0          324211     M           1985.0  2020-01-01 06:02:22   
2  14420219.0          611633     M           1981.0  2020-01-01 06:03:01   
3  14420220.0          521601     M           1989.0  2020-01-01 06:05:48   
4  14420221.0          631227     M           1979.0  2020-01-01 06:08:15   

            fin_trajet  id_depart  id_arrivee  
0  2020-01-01 06:05:38         52         268  
1  2020-01-01 06:07:32        254         180  
2  2020-01-01 06:21:43        258         278  
3  2020-01-01 06:15:11        200         201  
4  2020-01-01 06:29:33        151         248  
id_trajet                  float64
id_utilisateur               int32
genre              string[pyarrow]
annee_naissance            float64
debut_trajet       string[pyarrow]
fin_trajet         string[pyarrow]
id_depart

Récupérer les données des stations pour les joindre aux données

In [7]:
# Mettre en place la première jointure sur l'id de départ
ddf = ddf.merge(
    df_stations,
    left_on='id_depart',  # Clé de jointure dans le DataFrame des trajets
    right_on='station_id', # Clé de jointure dans le DataFrame des stations
    how='left',           # Conserver tous les trajets même si l'ID de station est manquant
    suffixes=('', '_depart') # Ajoute le suffixe '_depart' aux colonnes de stations
)

# Renommer les colonnes de coordonnées et de nom pour le départ
ddf = ddf.rename(columns={
    'nom': 'nom_station_depart',
    'lieu': 'quartier_depart',
    'latitude': 'lat_depart',
    'longitude': 'lon_depart'
})

# Supprimer la colonne station_id introduite par la jointure
ddf = ddf.drop(columns=['station_id'])


# Placer la colonne id_destination à la fin

ddf = ddf[[col for col in ddf.columns if col != 'id_arrivee'] + ['id_arrivee']]

# Mettre en place la seconde jointure sur l'id d'arrivée
ddf = ddf.merge(
    df_stations,
    left_on='id_arrivee',  # Clé de jointure dans le DataFrame des trajets
    right_on='station_id', # Clé de jointure dans le DataFrame des stations
    how='left',           # Conserver tous les trajets même si l'ID de station est manquant
    suffixes=('', '_arrivee') # Ajoute le suffixe '_arrivee' aux colonnes de stations
)

# Renommer les colonnes de coordonnées et de nom pour le départ
ddf = ddf.rename(columns={
    'nom': 'nom_station_arrivee',
    'lieu': 'quartier_arrivee',
    'latitude': 'lat_arrivee',
    'longitude': 'lon_arrivee'
})

# Supprimer la colonne station_id introduite par la jointure
ddf = ddf.drop(columns=['station_id'])



In [8]:
ddf.head()

,id_trajet,id_utilisateur,genre,annee_naissance,debut_trajet,fin_trajet,id_depart,nom_station_depart,quartier_depart,lat_depart,lon_depart,id_arrivee,nom_station_arrivee,quartier_arrivee,lat_arrivee,lon_arrivee
0,14420217.0,451617,M,1992.0,2020-01-01 06:02:20,2020-01-01 06:05:38,52,(GDL-050) C. Pedro Moreno / Calz. Federalismo,POLÍGONO CENTRAL,20.675747,-103.354576,268,(GDL-195) C. Ramón Corona / Av. Juárez,POLÍGONO CENTRAL,20.675289,-103.346558
1,14420218.0,324211,M,1985.0,2020-01-01 06:02:22,2020-01-01 06:07:32,254,(GDL-020) C. General Coronado / C.Juan Manuel,POLÍGONO CENTRAL,20.678480,-103.365105,180,(GDL-113) C. Ghilardi / C. Arista,POLÍGONO CENTRAL,20.688034,-103.362717
2,14420219.0,611633,M,1981.0,2020-01-01 06:03:01,2020-01-01 06:21:43,258,(GDL-185) Dr. Silverio García /Av. Revolución,TLQ-CORREDORATLAS,20.663179,-103.329422,278,(GDL-205) Av. Agustín Yáñez /C. Venezuela,POLÍGONO CENTRAL,20.662649,-103.365540
3,14420220.0,521601,M,1989.0,2020-01-01 06:05:48,2020-01-01 06:15:11,200,(GDL-133) C. J. Ruíz de Alarcón / Av. La Paz,POLÍGONO CENTRAL,20.671890,-103.376083,201,(GDL-134) C. Agustín Yáñez / C. Fco Ugarte,POLÍGONO CENTRAL,20.670210,-103.384224
4,14420221.0,631227,M,1979.0,2020-01-01 06:08:15,2020-01-01 06:29:33,151,(TLQ-010) C. Zaragoza / C. Emilio Carranza,TLQ-CORREDORATLAS,20.638340,-103.309517,248,(GDL-180) C. Montes Pirineos/Salvador Quevedo,POLÍGONO CENTRAL,20.686920,-103.334671


In [9]:
# Utilisation de dd.to_datetime avec 'mixed' pour gérer les formats "MM/DD/YYYY" et "YYYY-MM-DD"
for col in ["debut_trajet", "fin_trajet"]:
    ddf[col] = dd.to_datetime(ddf[col], format='mixed', errors='coerce') 
    # 'errors='coerce' convertira les dates non valides en NaT (Not a Time)

In [10]:
# La conversion des dates transforment ces colonnes en float au lieu de int, on applique un correctif
cols_to_fix = ["id_trajet", "id_utilisateur", "id_depart", "id_arrivee"]

for col in cols_to_fix:
    ddf[col] = ddf[col].astype('int32')

In [11]:
# Supprimer les lignes où la date de début de trajet est manquante
ddf = ddf.dropna(subset=['debut_trajet']) 
# Supprimer les lignes où l'âge de naissance est manquant
ddf = ddf.dropna(subset=['annee_naissance'])
# Supprimer les lignes où les infos de stations sont manquantes
ddf = ddf.dropna(subset=['nom_station_depart'])
ddf = ddf.dropna(subset=['nom_station_arrivee'])


# Conversion de l'année de naissance en entier (puisque les NA sont nettoyés)
# Nous devons utiliser un entier qui supporte les NA (Int64 de Pandas)
ddf['annee_naissance'] = ddf['annee_naissance'].astype('Int64')

print("\n--- Dask DataFrame après nettoyage (lazy) ---")
print(ddf.head())
print(ddf.dtypes)


--- Dask DataFrame après nettoyage (lazy) ---
   id_trajet  id_utilisateur genre  annee_naissance        debut_trajet  \
0   14420217          451617     M             1992 2020-01-01 06:02:20   
1   14420218          324211     M             1985 2020-01-01 06:02:22   
2   14420219          611633     M             1981 2020-01-01 06:03:01   
3   14420220          521601     M             1989 2020-01-01 06:05:48   
4   14420221          631227     M             1979 2020-01-01 06:08:15   

           fin_trajet  id_depart  \
0 2020-01-01 06:05:38         52   
1 2020-01-01 06:07:32        254   
2 2020-01-01 06:21:43        258   
3 2020-01-01 06:15:11        200   
4 2020-01-01 06:29:33        151   

                              nom_station_depart    quartier_depart  \
0  (GDL-050) C. Pedro Moreno / Calz. Federalismo   POLÍGONO CENTRAL   
1  (GDL-020) C. General Coronado / C.Juan Manuel   POLÍGONO CENTRAL   
2  (GDL-185) Dr. Silverio García /Av. Revolución  TLQ-CORREDORATLAS   
3

In [12]:
# Ajout d'une colonne année (nécessaire pour le calcul de l'âge)
ddf['annee'] = ddf.debut_trajet.dt.year

# Ajout d'une colonne âge
ddf['age'] = ddf["annee"] - ddf["annee_naissance"]

# Ajout d'une colonne temps de trajet convertie en minutes
ddf['temps_trajet'] = (ddf.fin_trajet - ddf.debut_trajet) / pd.Timedelta(minutes=1)

# Ajout d'une colonne heure de départ
ddf['heure_debut'] = ddf.debut_trajet.dt.hour
ddf['heure_debut'] = ddf['heure_debut'].replace(0, 24) # On remplace 0 par 24 pour minuit pour l'affichage des graphiques

# Ajout d'une colonne mois et jour (pour l'analyse temporelle)
ddf['mois'] = ddf.debut_trajet.dt.strftime('%B')
ddf['jour'] = ddf.debut_trajet.dt.strftime('%A')

print("\n--- Dask DataFrame après ajout de colonnes (lazy) ---")
print(ddf.head())
print(ddf.dtypes)


--- Dask DataFrame après ajout de colonnes (lazy) ---
   id_trajet  id_utilisateur genre  annee_naissance        debut_trajet  \
0   14420217          451617     M             1992 2020-01-01 06:02:20   
1   14420218          324211     M             1985 2020-01-01 06:02:22   
2   14420219          611633     M             1981 2020-01-01 06:03:01   
3   14420220          521601     M             1989 2020-01-01 06:05:48   
4   14420221          631227     M             1979 2020-01-01 06:08:15   

           fin_trajet  id_depart  \
0 2020-01-01 06:05:38         52   
1 2020-01-01 06:07:32        254   
2 2020-01-01 06:21:43        258   
3 2020-01-01 06:15:11        200   
4 2020-01-01 06:29:33        151   

                              nom_station_depart    quartier_depart  \
0  (GDL-050) C. Pedro Moreno / Calz. Federalismo   POLÍGONO CENTRAL   
1  (GDL-020) C. General Coronado / C.Juan Manuel   POLÍGONO CENTRAL   
2  (GDL-185) Dr. Silverio García /Av. Revolución  TLQ-CORREDORAT

In [13]:
def haversine_vectorise(lon1, lat1, lon2, lat2):
    """
    Calcule la distance de Haversine entre deux jeux de coordonnées 
    (lon1, lat1) et (lon2, lat2). Les entrées sont des Séries Dask.
    Le résultat est en kilomètres (km).
    """
    # Rayon de la Terre en kilomètres (km)
    R = 6371 
    
    # Conversion des degrés en radians
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])

    # Différences de coordonnées
    dlon = lon2 - lon1
    dlat = lat2 - lat1

    # Formule de Haversine
    # Note: Dask utilise NumPy pour ces calculs vectoriels
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    # Distance finale
    distance_km = R * c
    return distance_km

# S'assurer que les coordonnées sont du type float pour le calcul
ddf['lat_depart'] = ddf['lat_depart'].astype('float64')
ddf['lon_depart'] = ddf['lon_depart'].astype('float64')
ddf['lat_arrivee'] = ddf['lat_arrivee'].astype('float64')
ddf['lon_arrivee'] = ddf['lon_arrivee'].astype('float64')


# Ajout de la colonne de distance de Haversine (en mode paresseux)
ddf['distance_km'] = haversine_vectorise(
    ddf['lon_depart'], 
    ddf['lat_depart'], 
    ddf['lon_arrivee'], 
    ddf['lat_arrivee']
)

print("--- Dask DataFrame après ajout de la colonne 'distance_km' ---")
print(ddf[['nom_station_depart', 'nom_station_arrivee', 'distance_km']].head())
print(ddf.dtypes['distance_km'])

--- Dask DataFrame après ajout de la colonne 'distance_km' ---
                              nom_station_depart  \
0  (GDL-050) C. Pedro Moreno / Calz. Federalismo   
1  (GDL-020) C. General Coronado / C.Juan Manuel   
2  (GDL-185) Dr. Silverio García /Av. Revolución   
3   (GDL-133) C. J. Ruíz de Alarcón / Av. La Paz   
4     (TLQ-010) C. Zaragoza / C. Emilio Carranza   

                             nom_station_arrivee  distance_km  
0         (GDL-195) C. Ramón Corona / Av. Juárez     0.835743  
1              (GDL-113) C. Ghilardi / C. Arista     1.091005  
2      (GDL-205) Av. Agustín Yáñez /C. Venezuela     3.758207  
3     (GDL-134) C. Agustín Yáñez / C. Fco Ugarte     0.867282  
4  (GDL-180) C. Montes Pirineos/Salvador Quevedo     6.002445  
float64


In [14]:
# Suppression des données dupliquées
ddf= ddf.drop_duplicates(
    subset=['id_trajet'], 
    keep='first'
)

Appliquer des filtres pour supprimer les valeurs aberrantes

In [15]:

# Filtre sur le temps de trajet (en minutes)
TEMPS_MIN_MINUTES = 1
TEMPS_MAX_MINUTES = 240 # 4 heures

ddf = ddf[
    (ddf['temps_trajet'] > TEMPS_MIN_MINUTES) & 
    (ddf['temps_trajet'] < TEMPS_MAX_MINUTES)
]

# Filtre sur la distance (en km)
# On ne filtre pas sur la distance minimale car si station_depart == station_arrivee, la distance est 0 malgré le trajet
DISTANCE_MAX_KM = 30 

ddf = ddf[
    (ddf['distance_km'] < DISTANCE_MAX_KM)
]

# Filtre sur les heures de service (il n'est possible d'emprunter un vélo qu'entre 5h et minuit)
heure_debut_service = 5
heure_fin_service = 24

ddf = ddf[
    (ddf["heure_debut"] >= heure_debut_service) &
    (ddf["heure_debut"] <= heure_fin_service)
]
    
# Pas de filtre d'âge à cette étape, elle sera réalisée ultérieurement
# En effet, bien que certaines valeurs soient aberrantes, les trajets ont bien eu lieu et il serait dommage de les supprimer'''

In [16]:
output_dir = 'cleaned_data_2020'

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
    print(f"Dossier '{output_dir}' supprimé.")
    
print("Déclenchement du calcul et écriture des données...")

# Utilisez compute=True par défaut pour que l'opération soit synchrone
ddf.to_parquet(
    path= output_dir,
    engine='pyarrow',  # Moteur de lecture/écriture recommandé pour la performance
    write_metadata_file=True, # Écrit le fichier _metadata pour des lectures Dask encore plus rapides
    compression='snappy' # Compression rapide et efficace
)

print(f"\n✅ Sauvegarde terminée. Les données sont disponibles dans le dossier")
print(f"Nombre de partitions écrites : {ddf.npartitions}")

Dossier 'cleaned_data_2020' supprimé.
Déclenchement du calcul et écriture des données...

✅ Sauvegarde terminée. Les données sont disponibles dans le dossier
Nombre de partitions écrites : 72
